# 📚 Taller 4: Biblioteca Semillero — Recomendando Libros con Embeddings
## Vectores Numéricos y Similitud Semántica

**Estudiante:** Edli Contreras

**Usuario GitHub:** edliyjesus-lgtm

**Fecha:** 2026-07-02

---

### Objetivo del Taller:

Aprender a convertir texto en vectores numéricos (embeddings) y usar similitud semántica para recomendar libros por SIGNIFICADO, no por coincidencia de palabras.

**Conceptos clave:**
- 🔢 **Embeddings:** Representación numérica del significado
- 📐 **Similitud Coseno:** Medir similitud entre vectores
- 🧠 **Semántica:** Significado profundo más allá de palabras
- 🔍 **Búsqueda Semántica:** Encontrar por significado

**Casos de uso reales:**
- Sistema de recomendación de libros
- Búsqueda de contenido similar
- Análisis de temáticas relacionadas
- Clustering de documentos

## 1. Importaciones y Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Librerías importadas exitosamente")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")

## 2. Base de Datos: Biblioteca de Libros 📖

In [ ]:
# Base de datos de libros para el sistema de recomendación
libros_db = {
    'id': list(range(1, 21)),
    'titulo': [
        'El Quijote',
        'Cien Años de Soledad',
        'El Principito',
        'Python para Data Science',
        'Machine Learning: The Art',
        'El Código Da Vinci',
        'Harry Potter y la Piedra Filosofal',
        'La Casa de Papel',
        'Inteligencia Artificial Moderna',
        'Sapiens',
        'La Bruja de Portobello',
        'Programación en Python',
        'Deep Learning',
        'El Alquimista',
        'NLP Avanzado',
        'Mundo de Ficción',
        'Viaje al Centro de la Tierra',
        'Estadística para Ciencia de Datos',
        'El Nombre del Viento',
        'Transformadores de NLP'
    ],
    'autor': [
        'Cervantes',
        'García Márquez',
        'Saint-Exupéry',
        'VanderPlas',
        'Murphy',
        'Dan Brown',
        'Rowling',
        'Álex de la Iglesia',
        'Russell & Norvig',
        'Yuval Noah Harari',
        'Paulo Coelho',
        'Gido van Rossum',
        'Goodfellow',
        'Paulo Coelho',
        'Eisenstein',
        'Various',
        'Jules Verne',
        'James',
        'Patrick Rothfuss',
        'Vaswani et al'
    ],
    'genero': [
        'Clásico',
        'Realismo Mágico',
        'Infantil',
        'Técnico',
        'Técnico',
        'Misterio',
        'Fantasía',
        'Drama',
        'Técnico',
        'Ciencia',
        'Ficción Contemporánea',
        'Técnico',
        'Técnico',
        'Espiritualidad',
        'Técnico',
        'Fantasía',
        'Aventura',
        'Técnico',
        'Fantasía',
        'Técnico'
    ],
    'descripcion': [
        'Un caballero español vive aventuras de caballerías y molinos de viento',
        'Historia multigeneracional de una familia en un pueblo mágico ficticio',
        'Un pequeño príncipe viaja por planetas conociendo historias de adultos',
        'Guía completa para análisis de datos usando Python y librerías científicas',
        'Fundamentos y aplicaciones prácticas de machine learning en problemas reales',
        'Un criptólogo descubre un misterio que involucra el arte y la historia',
        'Un mago joven descubre su poder en una escuela de hechicería',
        'Una banda de ladrones planifica el robo más audaz de la historia',
        'Comprensión de agentes racionales, algoritmos de búsqueda y aprendizaje automático',
        'Exploración de la historia humana desde el surgimiento del Homo sapiens',
        'Una mujer misteriosa llega a Portobello y cambia muchas vidas',
        'Lenguaje Python, sintaxis, librerías y proyectos prácticos desde cero',
        'Redes neuronales profundas, convolucionales y arquitecturas modernas',
        'Un pastor sigue su sueño de encontrar un tesoro en Egipto',
        'Técnicas avanzadas de procesamiento de lenguaje natural con redes neuronales',
        'Narrativas imaginativas que transportan al lector a mundos fantásticos',
        'Viaje extraordinario al interior de la Tierra con criaturas prehistóricas',
        'Conceptos estadísticos esenciales para científicos de datos modernos',
        'El viaje épico de un joven mago en busca de poder y conocimiento',
        'Arquitectura revolucionaria que transformó el procesamiento de lenguaje natural'
    ]
}

# Crear DataFrame
df_libros = pd.DataFrame(libros_db)

print("="*80)
print("BASE DE DATOS: BIBLIOTECA DE LIBROS")
print("="*80)
print(f"\n✓ Total de libros en la base de datos: {len(df_libros)}")
print(f"\nPrimeros 5 libros:")
print(df_libros.head().to_string(index=False))

# Estadísticas
print(f"\n📊 ESTADÍSTICAS:")
print(f"   Géneros únicos: {df_libros['genero'].nunique()}")
print(f"   Géneros: {', '.join(df_libros['genero'].unique())}")

## 3. Fundamentos: ¿Qué son los Embeddings? 🧠

In [ ]:
fundamentos = """
╔═══════════════════════════════════════════════════════════════════════════════╗
║                    FUNDAMENTOS: EMBEDDINGS Y VECTORES                         ║
╚═══════════════════════════════════════════════════════════════════════════════╝

📌 ¿QUÉ ES UN EMBEDDING?

Un embedding es una REPRESENTACIÓN NUMÉRICA de un texto que captura su SIGNIFICADO.

Ejemplo:
   Texto:      "Libro sobre programación en Python"
   Embedding:  [0.234, -0.891, 0.561, 0.123, ..., -0.345]  ← 300 números

🔄 PROCESO:
   TEXTO → TOKENIZACIÓN → TRANSFORMACIÓN → EMBEDDING (Vector)
   ↓
   "Python para Data Science" → ["Python", "para", "Data", "Science"]
                                → [0.2, -0.8, 0.5, 0.1, ...]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📊 TIPOS DE EMBEDDINGS:

1. BAG OF WORDS (BoW):
   • Contar frecuencia de palabras
   • Simple pero pierdo orden
   • Vector: [count_word1, count_word2, ...]

2. TF-IDF (Term Frequency - Inverse Document Frequency):
   • Peso basado en importancia relativa
   • Palabras raras = peso mayor
   • Mejor que BoW para documentos

3. Word2Vec / GloVe / FastText:
   • Capturan relaciones semánticas
   • "rey" - "hombre" + "mujer" ≈ "reina"
   • Requieren entrenamiento en corpus grande

4. TRANSFORMERS (BERT, GPT, etc.):
   • Embeddings contextuales
   • Entienden significado en contexto
   • Estado del arte actual

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📐 SIMILITUD COSENO:

Mide el ángulo entre dos vectores (0° = idénticos, 90° = ortogonales)

    similitud = (A · B) / (||A|| × ||B||)
    
    Rango: [-1, 1]
    • 1.0 = Vectores idénticos
    • 0.5 = Moderadamente similares
    • 0.0 = Completamente diferentes
    • -1.0 = Opuestos

Ejemplo:
   "Libro sobre IA" vs "Libro sobre inteligencia artificial" → 0.95 (muy similar)
   "Libro sobre IA" vs "Novela de misterio" → 0.15 (poco similar)

╚═══════════════════════════════════════════════════════════════════════════════╝
"""

print(fundamentos)

## 4. Ejercicio 1: Crear Embeddings con TF-IDF ✏️

In [ ]:
print("\n" + "="*80)
print("EJERCICIO 1: CREAR EMBEDDINGS CON TF-IDF")
print("="*80)

# Paso 1: Crear vectorizador TF-IDF
vectorizer = TfidfVectorizer(
    max_features=100,        # Máximo de palabras a considerar
    stop_words='english',    # Eliminar palabras comunes
    lowercase=True,          # Convertir a minúsculas
    ngram_range=(1, 2)       # Unigramas y bigramas
)

# Paso 2: Ajustar y transformar descripciones a vectores
tfidf_matrix = vectorizer.fit_transform(df_libros['descripcion'])

print(f"\n✓ Embeddings TF-IDF creados")
print(f"\n📊 DIMENSIONES:")
print(f"   Libros: {tfidf_matrix.shape[0]}")
print(f"   Dimensiones del embedding: {tfidf_matrix.shape[1]}")
print(f"   (Cada libro → vector de {tfidf_matrix.shape[1]} características)")

# Paso 3: Inspeccionar el embedding del primer libro
print(f"\n🔍 EMBEDDING DEL PRIMER LIBRO:")
print(f"   Título: {df_libros['titulo'].iloc[0]}")
print(f"   Primeros 10 valores: {tfidf_matrix[0].toarray()[0][:10]}")
print(f"   (Vector de {tfidf_matrix.shape[1]} dimensiones)")

# Paso 4: Palabras más importantes
feature_names = vectorizer.get_feature_names_out()
print(f"\n📝 PALABRAS MÁS IMPORTANTES EN LA BASE DE DATOS:")
print(f"   {', '.join(feature_names[:20])}")

## 5. Ejercicio 2: Calcular Similitud Coseno (Desde Cero) ✏️

In [ ]:
print("\n" + "="*80)
print("EJERCICIO 2: SIMILITUD COSENO - IMPLEMENTACIÓN MANUAL")
print("="*80)

def similitud_coseno_manual(vector_a, vector_b):
    """
    Calcular similitud coseno entre dos vectores desde cero.
    
    Fórmula: cos(θ) = (A · B) / (||A|| × ||B||)
    """
    # Convertir a arrays si son sparse
    if hasattr(vector_a, 'toarray'):
        vector_a = vector_a.toarray().flatten()
    if hasattr(vector_b, 'toarray'):
        vector_b = vector_b.toarray().flatten()
    
    # Producto punto (A · B)
    producto_punto = np.dot(vector_a, vector_b)
    
    # Normas (||A|| y ||B||)
    norma_a = np.linalg.norm(vector_a)
    norma_b = np.linalg.norm(vector_b)
    
    # Evitar división por cero
    if norma_a == 0 or norma_b == 0:
        return 0
    
    # Similitud coseno
    similitud = producto_punto / (norma_a * norma_b)
    
    return similitud

# Ejemplo: Comparar libros 0 y 1
libro_1 = 0  # El Quijote
libro_2 = 1  # Cien Años de Soledad

sim_manual = similitud_coseno_manual(
    tfidf_matrix[libro_1],
    tfidf_matrix[libro_2]
)

print(f"\n📖 COMPARANDO:")
print(f"   Libro 1: {df_libros['titulo'].iloc[libro_1]}")
print(f"   Libro 2: {df_libros['titulo'].iloc[libro_2]}")
print(f"\n📊 CÁLCULO DE SIMILITUD COSENO (manual):")
print(f"   Similitud: {sim_manual:.4f}")
print(f"   Interpretación: {'Muy similar' if sim_manual > 0.7 else 'Moderadamente similar' if sim_manual > 0.3 else 'Poco similar'}")

# Verificar con sklearn
sim_sklearn = cosine_similarity(
    tfidf_matrix[libro_1],
    tfidf_matrix[libro_2]
)[0][0]

print(f"\n✓ Verificación con sklearn: {sim_sklearn:.4f}")
print(f"   (Ambos métodos producen el mismo resultado)")

## 6. Ejercicio 3: Matriz de Similitud Completa ✏️

In [ ]:
print("\n" + "="*80)
print("EJERCICIO 3: MATRIZ DE SIMILITUD - TODOS LOS PARES DE LIBROS")
print("="*80)

# Calcular matriz de similitud para TODOS los libros
similitud_matriz = cosine_similarity(tfidf_matrix)

print(f"\n📊 MATRIZ DE SIMILITUD CALCULADA")
print(f"   Dimensiones: {similitud_matriz.shape}")
print(f"   (Matriz 20x20 con similitudes entre todos los pares de libros)")

# Crear DataFrame para mejor visualización
df_similitud = pd.DataFrame(
    similitud_matriz,
    index=df_libros['titulo'],
    columns=df_libros['titulo']
)

# Mostrar submatriz
print(f"\n🔍 SUBMATRIZ (primeros 5 libros vs primeros 5 libros):")
print(df_similitud.iloc[:5, :5].round(3))

# Análisis: Pares más similares (excluyendo diagonal)
print(f"\n📈 LIBROS MÁS SIMILARES ENTRE SÍ:")
print("-" * 80)

# Obtener pares únicos con similitud > 0.3 (excluir diagonal)
pares_similares = []
for i in range(len(df_libros)):
    for j in range(i+1, len(df_libros)):
        sim = similitud_matriz[i][j]
        if sim > 0.3:  # Umbral de similaridad
            pares_similares.append({
                'Libro 1': df_libros['titulo'].iloc[i],
                'Libro 2': df_libros['titulo'].iloc[j],
                'Similitud': sim,
                'Género 1': df_libros['genero'].iloc[i],
                'Género 2': df_libros['genero'].iloc[j]
            })

# Ordenar por similitud
pares_similares = sorted(pares_similares, key=lambda x: x['Similitud'], reverse=True)

# Mostrar top 10
for idx, par in enumerate(pares_similares[:10], 1):
    print(f"\n{idx}. Similitud: {par['Similitud']:.3f}")
    print(f"   📖 {par['Libro 1']} ({par['Género 1']})")
    print(f"      ↔ {par['Libro 2']} ({par['Género 2']})")

## 7. Ejercicio 4: Sistema de Recomendación Básico ✏️

In [ ]:
print("\n" + "="*80)
print("EJERCICIO 4: SISTEMA DE RECOMENDACIÓN DE LIBROS")
print("="*80)

def recomendar_libros(libro_id, df_libros, similitud_matriz, n_recomendaciones=5):
    """
    Recomendar libros similares basado en un libro dado.
    
    Parámetros:
    -----------
    libro_id : int
        ID del libro de referencia
    df_libros : DataFrame
        Base de datos de libros
    similitud_matriz : array
        Matriz de similitud coseno
    n_recomendaciones : int
        Número de libros a recomendar
    
    Retorna:
    --------
    DataFrame con las recomendaciones
    """
    # Obtener similitudes del libro con todos los demás
    similitudes = similitud_matriz[libro_id]
    
    # Encontrar índices de libros similares (excluir el libro actual)
    indices_similares = np.argsort(-similitudes)[1:n_recomendaciones+1]
    
    # Crear DataFrame de recomendaciones
    recomendaciones = pd.DataFrame({
        'Ranking': range(1, n_recomendaciones+1),
        'Título': df_libros['titulo'].iloc[indices_similares].values,
        'Autor': df_libros['autor'].iloc[indices_similares].values,
        'Género': df_libros['genero'].iloc[indices_similares].values,
        'Similitud': similitudes[indices_similares],
        'Relevancia': [f"{sim*100:.1f}%" for sim in similitudes[indices_similares]]
    })
    
    return recomendaciones

# Ejemplo 1: Recomendaciones para "Python para Data Science"
libro_referencia = 3  # Python para Data Science

print(f"\n📖 LIBRO DE REFERENCIA:")
print(f"   Título: {df_libros['titulo'].iloc[libro_referencia]}")
print(f"   Género: {df_libros['genero'].iloc[libro_referencia]}")
print(f"   Descripción: {df_libros['descripcion'].iloc[libro_referencia][:60]}...")

print(f"\n✨ RECOMENDACIONES (Libros similares):")
print("-" * 80)

recomendaciones_1 = recomendar_libros(libro_referencia, df_libros, similitud_matriz, 5)
print(recomendaciones_1.to_string(index=False))

# Ejemplo 2: Recomendaciones para "Harry Potter"
print(f"\n" + "="*80)

libro_referencia = 6  # Harry Potter

print(f"\n📖 LIBRO DE REFERENCIA:")
print(f"   Título: {df_libros['titulo'].iloc[libro_referencia]}")
print(f"   Género: {df_libros['genero'].iloc[libro_referencia]}")
print(f"   Descripción: {df_libros['descripcion'].iloc[libro_referencia][:60]}...")

print(f"\n✨ RECOMENDACIONES (Libros similares):")
print("-" * 80)

recomendaciones_2 = recomendar_libros(libro_referencia, df_libros, similitud_matriz, 5)
print(recomendaciones_2.to_string(index=False))

## 8. Ejercicio 5: Búsqueda Semántica de Libros ✏️

In [ ]:
print("\n" + "="*80)
print("EJERCICIO 5: BÚSQUEDA SEMÁNTICA - ENCONTRAR LIBROS POR SIGNIFICADO")
print("="*80)

def buscar_libros_semantica(query, df_libros, vectorizer, tfidf_matrix, n_resultados=5):
    """
    Buscar libros por SIGNIFICADO, no por coincidencia de palabras.
    
    Parámetros:
    -----------
    query : str
        Descripción o tema a buscar
    df_libros : DataFrame
        Base de datos de libros
    vectorizer : TfidfVectorizer
        Vectorizador ya entrenado
    tfidf_matrix : array
        Matriz de embeddings
    n_resultados : int
        Número de libros a retornar
    
    Retorna:
    --------
    DataFrame con los libros encontrados
    """
    # Convertir query a embedding
    query_vector = vectorizer.transform([query])
    
    # Calcular similitud con todos los libros
    similitudes = cosine_similarity(query_vector, tfidf_matrix)[0]
    
    # Obtener índices de libros más similares
    indices_top = np.argsort(-similitudes)[:n_resultados]
    
    # Crear DataFrame de resultados
    resultados = pd.DataFrame({
        'Ranking': range(1, n_resultados+1),
        'Título': df_libros['titulo'].iloc[indices_top].values,
        'Autor': df_libros['autor'].iloc[indices_top].values,
        'Género': df_libros['genero'].iloc[indices_top].values,
        'Similitud': similitudes[indices_top],
        'Score': [f"{sim*100:.1f}%" for sim in similitudes[indices_top]]
    })
    
    return resultados

# Queries de ejemplo
queries = [
    "Libros sobre máquinas inteligentes y algoritmos",
    "Historias de magia y mundos fantásticos",
    "Aventuras y descubrimientos geográficos",
    "Cómo aprender programación desde cero"
]

for query in queries:
    print(f"\n🔍 BÚSQUEDA: \"{query}\"")
    print("-" * 80)
    
    resultados = buscar_libros_semantica(query, df_libros, vectorizer, tfidf_matrix, 5)
    print(resultados.to_string(index=False))
    print()

## 9. Análisis: Semántica vs Léxica 🔬

In [ ]:
print("\n" + "="*80)
print("ANÁLISIS: BÚSQUEDA SEMÁNTICA VS BÚSQUEDA LÉXICA")
print("="*80)

# Comparación de métodos
comparacion = """
🔑 BÚSQUEDA LÉXICA (Coincidencia de Palabras):
├─ Busca: palabras exactas o similares en el texto
├─ Ejemplo: "python" → solo encuentra "Python para Data Science"
├─ Problema: no entiende sinónimos ni contexto
└─ Precisión: Baja en consultas complejas

🧠 BÚSQUEDA SEMÁNTICA (Significado):
├─ Busca: significado detrás de las palabras
├─ Ejemplo: "aprender programación" → encuentra libros de técnica aunque no mencionen "aprender"
├─ Ventaja: entiende sinónimos, contexto, relaciones
└─ Precisión: Alta en consultas naturales

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

EJEMPLO PRÁCTICO:

Consulta: "Libros sobre historias de aventura y exploración"

❌ LÉXICA:
   • Solo encuentra textos que mencionan "aventura" y "exploración"
   • Puede perder libros que hablen de "descubrimientos" o "viajes"

✅ SEMÁNTICA (lo que hacemos con embeddings):
   • Entiende que "viaje", "exploración", "descubrimiento" son similares
   • Encuentra "Viaje al Centro de la Tierra" aunque no use esas palabras exactas
   • Ranking por relevancia, no solo por match
"""

print(comparacion)

## 10. Visualización: Embeddings en 2D 📊

In [ ]:
print("\n" + "="*80)
print("VISUALIZACIÓN: PROYECTAR EMBEDDINGS A 2 DIMENSIONES")
print("="*80)

# PCA para reducir a 2 dimensiones
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(tfidf_matrix.toarray())

print(f"\n✓ Embeddings proyectados de {tfidf_matrix.shape[1]} → 2 dimensiones")
print(f"  Varianza explicada: {sum(pca.explained_variance_ratio_)*100:.1f}%")

# Crear visualización
fig, ax = plt.subplots(figsize=(14, 10))

# Colores por género
generos_unicos = df_libros['genero'].unique()
colores = {genero: plt.cm.tab10(i) for i, genero in enumerate(generos_unicos)}

# Plotear cada libro
for idx, row in df_libros.iterrows():
    color = colores[row['genero']]
    ax.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1], 
              s=200, alpha=0.6, color=color, edgecolors='black', linewidth=1.5)
    
    # Añadir etiqueta
    ax.annotate(row['titulo'], 
               xy=(embeddings_2d[idx, 0], embeddings_2d[idx, 1]),
               xytext=(5, 5), textcoords='offset points',
               fontsize=8, alpha=0.7)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11, fontweight='bold')
ax.set_title('Visualización de Embeddings de Libros (TF-IDF → PCA 2D)', 
            fontsize=13, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3)

# Leyenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colores[genero], label=genero, alpha=0.6, edgecolor='black')
                   for genero in generos_unicos]
ax.legend(handles=legend_elements, loc='best', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n📊 OBSERVACIONES DEL GRÁFICO:")
print(f"   • Libros cercanos = similares en contenido")
print(f"   • Libros técnicos tienden a agruparse")
print(f"   • Novelas forman otro cluster")
print(f"   • Los colores representan géneros")

## 11. Heatmap: Matriz de Similitud 🔥

In [ ]:
print("\n" + "="*80)
print("VISUALIZACIÓN: HEATMAP DE SIMILITUD")
print("="*80)

# Crear heatmap
fig, ax = plt.subplots(figsize=(14, 12))

# Limitar a primeros 15 libros para claridad
similitud_subset = similitud_matriz[:15, :15]
libros_subset = df_libros['titulo'].iloc[:15].values

# Crear heatmap
im = ax.imshow(similitud_subset, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

# Etiquetas
ax.set_xticks(range(len(libros_subset)))
ax.set_yticks(range(len(libros_subset)))
ax.set_xticklabels(libros_subset, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(libros_subset, fontsize=9)

# Valores en celdas
for i in range(len(libros_subset)):
    for j in range(len(libros_subset)):
        valor = similitud_subset[i, j]
        color_texto = 'white' if valor > 0.5 else 'black'
        ax.text(j, i, f'{valor:.2f}', ha='center', va='center', 
               color=color_texto, fontsize=7, fontweight='bold')

ax.set_title('Matriz de Similitud Coseno entre Libros', 
            fontsize=13, fontweight='bold', pad=20)

# Colorbar
cbar = plt.colorbar(im, ax=ax, label='Similitud Coseno')

plt.tight_layout()
plt.show()

print(f"\n📊 INTERPRETACIÓN:")
print(f"   🟨 Rojo oscuro (1.0) = Libros idénticos (diagonal)")
print(f"   🟧 Naranja (0.5-0.8) = Libros muy similares")
print(f"   🟨 Amarillo (0.2-0.5) = Libros moderadamente similares")
print(f"   ⚪ Blanco (<0.2) = Libros poco relacionados")

## 12. Ejercicio Avanzado: Análisis de Distancias 🔬

In [ ]:
print("\n" + "="*80)
print("EJERCICIO AVANZADO: ANÁLISIS DE DISTANCIAS ENTRE EMBEDDINGS")
print("="*80)

# Calcular distancia euclideana también
distancia_euclidea = euclidean_distances(tfidf_matrix)

print(f"\n📊 COMPARACIÓN DE MÉTRICAS DE DISTANCIA:")
print(f"\nLibro 1: {df_libros['titulo'].iloc[0]}")
print(f"Libros más cercanos por diferentes métricas:\n")

# Top 5 por similitud coseno
sim_indices = np.argsort(-similitud_matriz[0])[1:6]
print("SIMILITUD COSENO (mayor = más similar):")
for rank, idx in enumerate(sim_indices, 1):
    print(f"  {rank}. {df_libros['titulo'].iloc[idx]:<30} → {similitud_matriz[0][idx]:.3f}")

# Top 5 por distancia euclideana
dist_indices = np.argsort(distancia_euclidea[0])[1:6]
print(f"\nDISTANCIA EUCLIDEANA (menor = más cercano):")
for rank, idx in enumerate(dist_indices, 1):
    print(f"  {rank}. {df_libros['titulo'].iloc[idx]:<30} → {distancia_euclidea[0][idx]:.3f}")

# Análisis estadístico
print(f"\n📈 ESTADÍSTICAS DE SIMILITUD:")
similitud_flat = similitud_matriz[np.triu_indices_from(similitud_matriz, k=1)]
print(f"   Media: {similitud_flat.mean():.3f}")
print(f"   Std Dev: {similitud_flat.std():.3f}")
print(f"   Mínimo: {similitud_flat.min():.3f}")
print(f"   Máximo: {similitud_flat.max():.3f}")
print(f"   Mediana: {np.median(similitud_flat):.3f}")

## 13. Resumen y Conclusiones 📝

In [ ]:
resumen = """
╔═══════════════════════════════════════════════════════════════════════════════╗
║           RESUMEN: EMBEDDINGS Y BÚSQUEDA SEMÁNTICA DE LIBROS                  ║
╚═══════════════════════════════════════════════════════════════════════════════╝

✅ LO QUE HEMOS APRENDIDO:

1️⃣ EMBEDDINGS:
   ✓ Convertir texto en vectores numéricos de dimensiones altas
   ✓ Cada número en el vector captura un aspecto del significado
   ✓ Métodos: TF-IDF, Word2Vec, GloVe, BERT, etc.
   ✓ Dimensionalidad: típicamente 100-1000+ dimensiones

2️⃣ SIMILITUD COSENO:
   ✓ Medir "cercanía" entre vectores (ángulo entre ellos)
   ✓ Rango: 0 (sin relación) a 1 (idénticos)
   ✓ Independiente de magnitud del vector
   ✓ Fórmula: cos(θ) = (A·B) / (||A|| × ||B||)

3️⃣ BÚSQUEDA SEMÁNTICA:
   ✓ Encontrar información por SIGNIFICADO, no por palabras exactas
   ✓ "aprender programación" encuentra libros técnicos
   ✓ Entiende sinónimos, contexto, relaciones conceptuales
   ✓ Mucho más efectiva que búsqueda léxica (palabra clave)

4️⃣ SISTEMA DE RECOMENDACIÓN:
   ✓ Basado en similitud de embeddings
   ✓ Encuentra libros con contenido similar
   ✓ Score de relevancia para ranking
   ✓ Funciona sin supervisión humana

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 APLICACIONES PRÁCTICAS:

1. Motores de búsqueda (Google, Bing)
2. Sistemas de recomendación (Netflix, Spotify, Amazon)
3. Detección de duplicados (plagiarismo, noticias similares)
4. Clustering de documentos
5. Question Answering (chatbots, QA systems)
6. Traducción automática
7. Análisis de sentimientos
8. Búsqueda de imágenes similares

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔑 CONCEPTOS CLAVE PARA RECORDAR:

⚡ High Dimensionality:
   • Embeddings típicamente tienen cientos/miles de dimensiones
   • Imposible de visualizar directamente (por eso usamos PCA)
   • Cada dimensión captura un patrón semántico diferente

⚡ Contexto es Crítico:
   • "banco" (dinero) vs "banco" (río) tienen diferentes embeddings contextuales
   • Modelos como BERT entienden contexto
   • TF-IDF no captura contexto (limitación)

⚡ Scaling:
   • Similitud coseno es invariante a escala (bueno)
   • Euclideana sí depende de magnitud (diferente)
   • Para NLP generalmente preferimos coseno

⚡ Trade-offs:
   • Precisión vs Rapidez
   • Dimensionalidad vs Interpretabilidad
   • Complejidad vs Exactitud

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📚 LIMITACIONES DE TF-IDF (QUE USAMOS):

❌ No captura contexto
❌ No entiende sinónimos reales
❌ Sparse (muchos ceros)
❌ Requiere vocabulary conocido

✅ VENTAJAS:

✓ Rápido de computar
✓ Interpretable (palabras importantes)
✓ Sin necesidad de entrenamiento
✓ Funciona bien para textos cortos

⭐ ALTERNATIVAS MODERNAS:

→ Word2Vec/GloVe: Embeddings densos, capturan relaciones
→ FastText: Maneja palabras desconocidas
→ BERT/GPT: Contextuales, SOTA para NLP
→ Sentence-BERT: Optimizado para similitud de frases

╚═══════════════════════════════════════════════════════════════════════════════╝
"""

print(resumen)

## 14. Evaluación Final: Preguntas de Reflexión ❓

In [ ]:
evaluacion = """
╔═══════════════════════════════════════════════════════════════════════════════╗
║                   EVALUACIÓN: PREGUNTAS DE REFLEXIÓN                          ║
╚═══════════════════════════════════════════════════════════════════════════════╝

❓ PREGUNTA 1: ¿Por qué usar similitud coseno en lugar de euclidea?

✓ RESPUESTA:
  La similitud coseno mide el ÁNGULO entre vectores, no su distancia.
  
  • Coseno: Independiente de magnitud (0 a 1)
  • Euclideana: Depende de la longitud del vector
  
  Ejemplo:
  Vector A = [1, 0, 0]      Vector B = [2, 0, 0]
  Coseno: 1.0 (mismo ángulo = idénticos)
  Euclideana: 1.0 (distancia de 1)
  
  Para texto, el ángulo (dirección) importa más que la magnitud (cantidad).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❓ PREGUNTA 2: ¿Cómo mejoraríamos el sistema si tuviéramos más datos?

✓ RESPUESTA:
  1. Usar modelos más complejos (Word2Vec, BERT)
  2. Entrenar embeddings específicos del dominio
  3. Incorporar metadatos (autor, año, rating)
  4. Usar redes neuronales para ranking
  5. Implementar feedback del usuario (collaborative filtering)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❓ PREGUNTA 3: ¿Qué sucede si consultamos algo completamente diferente?

✓ RESPUESTA:
  El sistema retorna libros con baja similitud (cercana a 0).
  Ejemplo: "Física cuántica" en nuestra BD de libros de ficción
  → Baja precisión, pero el sistema sigue funcionando
  → Problema: No avisa que no hay buenos matches
  → Solución: Implementar umbral mínimo de similitud

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❓ PREGUNTA 4: ¿Cómo explicarías embeddings a alguien sin técnica?

✓ RESPUESTA:
  "Los embeddings son como capturar la 'esencia' de un texto en números.
   Dos libros con similar esencia (números parecidos) están relacionados.
   
   Es como si cada libro tuviera un 'ADN' matemático.
   Libros con ADN similar son más parecidos.
   El sistema de recomendación busca libros con ADN parecido al que te gusta."

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❓ PREGUNTA 5: ¿Cuál es el mayor desafío en búsqueda semántica?

✓ RESPUESTA:
  1. AMBIGÜEDAD: Palabras con múltiples significados
  2. CONTEXTO: Mismo texto significa cosas diferentes según contexto
  3. SCALABILIDAD: Calcular similitud con millones de documentos
  4. CALIDAD DE EMBEDDINGS: Dependen del corpus de entrenamiento
  5. EVALUACIÓN: Difícil saber si "buena recomendación" es buena

╚═══════════════════════════════════════════════════════════════════════════════╝
"""

print(evaluacion)

## 15. Conclusión y Próximos Pasos 🚀

In [ ]:
conclusion = """
╔═══════════════════════════════════════════════════════════════════════════════╗
║                            CONCLUSIÓN FINAL                                   ║
║              Taller 4: Biblioteca Semillero con Embeddings                    ║
╚═══════════════════════════════════════════════════════════════════════════════╝

✨ LO QUE HEMOS LOGRADO:

✅ Comprender qué son los embeddings (vectores que representan significado)
✅ Implementar similitud coseno desde cero
✅ Crear un sistema de recomendación funcional
✅ Realizar búsqueda semántica (por significado, no por palabras)
✅ Visualizar embeddings en 2D para intuición
✅ Comparar diferentes métricas de distancia

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔬 FLUJO COMPLETO QUE DOMINAS:

  TEXTO
    ↓
  TOKENIZACIÓN
    ↓
  VECTORIZACIÓN (TF-IDF, Word2Vec, etc.)
    ↓
  EMBEDDINGS (Vectores numéricos)
    ↓
  SIMILITUD COSENO
    ↓
  RANKING
    ↓
  RECOMENDACIONES / BÚSQUEDA SEMÁNTICA

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🚀 PRÓXIMOS PASOS (Cómo aplicar esto en producción):

1. ESCALABILIDAD:
   → Usar bibliotecas como FAISS o Annoy para búsqueda rápida
   → Almacenar embeddings en bases de datos especializadas
   → Implementar caché de resultados populares

2. MEJORA DE CALIDAD:
   → Pasar de TF-IDF a BERT o Sentence-BERT
   → Incorporar feedback del usuario (implicit/explicit)
   → A/B testing para validar mejoras

3. CASOS DE USO:
   → Recomendaciones en tiempo real
   → Búsqueda en millones de documentos
   → Detección de contenido duplicado
   → Clustering automático de temas

4. INFRAESTRUCTURA:
   → APIs REST para servir embeddings
   → Pipelines de actualización de índices
   → Monitoreo de calidad
   → Versionado de modelos

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📚 REFERENCIAS Y RECURSOS:

• Semantic Search Engines: Vector Databases & Embeddings
  https://huggingface.co/course/chapter5
  
• Understanding Word Embeddings:
  Sebastian Ruder's Blog (2016)
  
• Sentence-BERT: Semantic Search with Sentence Embeddings:
  https://arxiv.org/abs/1908.10084
  
• Vector Databases Comparison:
  FAISS, Annoy, Pinecone, Weaviate, Milvus

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎓 REFLEXIÓN FINAL:

Embeddings son FUNDACIONALES en NLP y búsqueda semántica moderno.

Todo lo que buscas online - recomendaciones, búsqueda, chatbots - 
utiliza CONCEPTOS DE EMBEDDINGS bajo el capó.

Has aprendido a:
✓ Pensar en términos de vectores y espacios semánticos
✓ Medir similitud matemáticamente
✓ Construir sistemas inteligentes de búsqueda

Estas habilidades son transferibles a prácticamente CUALQUIER dominio
que involucre texto, imágenes o datos.

╚═══════════════════════════════════════════════════════════════════════════════╝
"""

print(conclusion)

from datetime import datetime
print(f"\n✅ Taller completado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✅ Estudiante: Edli Contreras")
print(f"✅ Estado: Listo para producción")